# Retrieval Augmented Generation (RAG) with Langchain
*Using IBM Granite Models*

## In this notebook
This notebook contains instructions for performing Retrieval Augumented Generation (RAG). RAG is an architectural pattern that can be used to augment the performance of language models by recalling factual information from a knowledge base, and adding that information to the model query. The most common approach in RAG is to create dense vector representations of the knowledge base in order to retrieve text chunks that are semantically similar to a given user query.

RAG use cases include:
- Customer service: Answering questions about a product or service using facts from the product documentation.
- Domain knowledge: Exploring a specialized domain (e.g., finance) using facts from papers or articles in the knowledge base.
- News chat: Chatting about current events by calling up relevant recent news articles.

In its simplest form, RAG requires 3 steps:

- Initial setup:
  - Index knowledge-base passages for efficient retrieval. In this recipe, we take embeddings of the passages and store them in a vector database.
- Upon each user query:
  - Retrieve relevant passages from the database. In this recipe, we use an embedding of the query to retrieve semantically similar passages.
  - Generate a response by feeding retrieved passage into a large language model, along with the user query.

## Setting up the environment

### Python version

Ensure you are running Python 3.10 or 3.11.

In [1]:
import sys
assert sys.version_info >= (3, 10) and sys.version_info < (3, 12), "Use Python 3.10 or 3.11 to run this notebook."

### Install dependencies

Granite Kitchen comes with a bundle of dependencies that are required for notebooks. See the list of packages in its [`setup.py`](https://github.com/ibm-granite-community/granite-kitchen/blob/main/setup.py). 

In [2]:
! pip install "git+https://github.com/ibm-granite-community/granite-kitchen" "langchain-huggingface" "langchain-milvus" "wget"

  Cloning https://github.com/ibm-granite-community/granite-kitchen to /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-req-build-fj1lbwxx
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-granite-community/granite-kitchen /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-req-build-fj1lbwxx
  Resolved https://github.com/ibm-granite-community/granite-kitchen to commit 0f452af66934bde0314a437eb7bdac560c54f256
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/ibm-granite-community/utils to /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-install-z37f47cl/ibm-granite-community-utils_956c3f484f984c3dab564462d0e97a58
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-granite-community/utils /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-install-z37f47cl/ibm-granite-community-uti

### Serving the Granite AI model


This notebook requires IBM Granite models to be served by an AI model runtime so that the models can be invoked or called. This notebook can use a locally accessible [Ollama](https://github.com/ollama/ollama) server to serve the models, or the [Replicate](https://replicate.com) cloud service.

During the pre-work, you may have either started a local Ollama server on your computer, or setup Replicate access and obtained an [API token](https://replicate.com/account/api-tokens).

## Selecting System Components

### Choose your Embeddings Model

Specify the model to use for generating embedding vectors from text.

To use a model from a provider other than Huggingface, replace this code cell with one from [this Embeddings Model recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Embeddings_Models.ipynb).

In [3]:
from langchain_ollama import OllamaEmbeddings
embeddings_model = OllamaEmbeddings(model="granite-embedding:278m")

### Choose your Vector Database

Specify the database to use for storing and retrieving embedding vectors.

To connect to a vector database other than Milvus substitute this code cell with one from [this Vector Store recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Vector_Stores.ipynb).

In [4]:
from langchain_milvus import Milvus
import tempfile

db_file = tempfile.NamedTemporaryFile(prefix="milvus_br_ollama_emb", suffix=".db", delete=False).name
print(f"The vector database will be saved to {db_file}")

vector_db = Milvus(embedding_function=embeddings_model, connection_args={"uri": db_file}, auto_id=True)

The vector database will be saved to /var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/milvus_br_ollama_embc7v44rm7.db


## Select your model


Select a Granite model to use. Here we use a Langchain client to connect to the model. If there is a locally accessible Ollama server, we use an Ollama client to access the model. Otherwise, we use a Replicate client to access the model.

When using Replicate, if the `REPLICATE_API_TOKEN` environment variable is not set, or a `REPLICATE_API_TOKEN` Colab secret is not set, then the notebook will ask for your [Replicate API token](https://replicate.com/account/api-tokens) in a dialog box.

In [5]:
import os
import requests
from langchain_ollama.llms import OllamaLLM
from langchain_community.llms import Replicate
from ibm_granite_community.notebook_utils import set_env_var

try: # Look for a locally accessible Ollama server for the model
    response = requests.get(os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434"))
    model = OllamaLLM(model="granite3.1-dense:8b")
except Exception: # Use Replicate for the model
    set_env_var("REPLICATE_API_TOKEN")
    model = Replicate(model="ibm-granite/granite-3.1-8b-instruct")


## Building the Vector Database

In this example, we take the State of the Union speech text, split it into chunks, derive embedding vectors using the embedding model, and load it into the vector database for querying.

### Download the document

Here we use President Biden's State of the Union address from March 1, 2022.

In [8]:
# import wget

# filename = 'silvio_santos.pdf'
# url = 'https://pt.wikipedia.org/w/index.php?title=Especial:DownloadAsPdf&page=Silvio_Santos&action=show-download-screen'

# if not os.path.isfile(filename):
#   wget.download(url, out=filename)

### Split the document into chunks

Split the document into text segments that can fit into the model's context window.

In [6]:
! pip install unstructured

from langchain.document_loaders import UnstructuredURLLoader
from langchain.text_splitter import CharacterTextSplitter

loader = UnstructuredURLLoader([ "https://pt.wikipedia.org/wiki/Silvio_Santos" ])
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

Created a chunk of size 841, which is longer than the specified 300
Created a chunk of size 819, which is longer than the specified 300
Created a chunk of size 1173, which is longer than the specified 300
Created a chunk of size 720, which is longer than the specified 300
Created a chunk of size 770, which is longer than the specified 300
Created a chunk of size 774, which is longer than the specified 300
Created a chunk of size 301, which is longer than the specified 300
Created a chunk of size 649, which is longer than the specified 300
Created a chunk of size 746, which is longer than the specified 300
Created a chunk of size 370, which is longer than the specified 300
Created a chunk of size 492, which is longer than the specified 300
Created a chunk of size 786, which is longer than the specified 300
Created a chunk of size 388, which is longer than the specified 300
Created a chunk of size 305, which is longer than the specified 300
Created a chunk of size 597, which is longer th

### Populate the vector database

NOTE: Population of the vector database may take over a minute depending on your embedding model and service.

In [7]:
vector_db.add_documents(texts)

[455178137904611328,
 455178137904611329,
 455178137904611330,
 455178137904611331,
 455178137904611332,
 455178137904611333,
 455178137904611334,
 455178137904611335,
 455178137904611336,
 455178137904611337,
 455178137904611338,
 455178137904611339,
 455178137904611340,
 455178137904611341,
 455178137904611342,
 455178137904611343,
 455178137904611344,
 455178137904611345,
 455178137904611346,
 455178137904611347,
 455178137904611348,
 455178137904611349,
 455178137904611350,
 455178137904611351,
 455178137904611352,
 455178137904611353,
 455178137904611354,
 455178137904611355,
 455178137904611356,
 455178137904611357,
 455178137904611358,
 455178137904611359,
 455178137904611360,
 455178137904611361,
 455178137904611362,
 455178137904611363,
 455178137904611364,
 455178137904611365,
 455178137904611366,
 455178137904611367,
 455178137904611368,
 455178137904611369,
 455178137904611370,
 455178137904611371,
 455178137904611372,
 455178137904611373,
 455178137904611374,
 455178137904

## Querying the Vector Database

### Conduct a similarity search

Search the database for similar documents by proximity of the embedded vector in vector space.

In [8]:
query = "Quando Silvio Santos morreu?"
docs = vector_db.similarity_search(query)
print(docs[0].page_content)

Silvio Santos, nome artístico de Senor Abravanel[7][8] (em hebraico: סניור אברבנאל;[9] Rio de Janeiro, 12 de dezembro de 1930 – São Paulo, 17 de agosto de 2024),[2] foi um apresentador de televisão, empresário, produtor, radialista e cantor brasileiro, conhecido como o "Rei da Televisão Brasileira",[10] além de ser amplamente reconhecido como a maior referência na história da comunicação do país.[11][12][13] Iniciou sua carreira de comunicador na rádio, posteriormente chegando a televisão, onde consagrou-se como uma figura central na cultura popular e entretenimento brasileiro por mais de seis décadas, conquistando o carinho do público e a expansão de seus negócios, através da fundação do Grupo Silvio Santos. Ao longo da sua vida, Silvio também esteve envolvido em outras áreas como a música e a política.[14]


## Answering Questions

### Automate the RAG pipeline

Build a RAG chain with the model and the document retriever. See the [Granite Prompting Guide](https://www.ibm.com/granite/docs/models/granite/#prompt-anatomy) for information on the prompt template.

In [9]:
from langchain.prompts import PromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Create a prompt template for question-answering with the retrieved context.
prompt_template = """\
<|start_of_role|>system<|end_of_role|>Você é um assistente brasileiro. Responda sempre em Português.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>Use os seguintes pedaços de contexto para responder à pergunta no final.
Se você não sabe a resposta, apenas diga que não sabe, não tente inventar uma resposta. 

{context}

Pergunta: {input}<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>"""

# Assemble the retrieval-augmented generation chain.
qa_chain_prompt = PromptTemplate.from_template(prompt_template)
combine_docs_chain = create_stuff_documents_chain(model, qa_chain_prompt)
rag_chain = create_retrieval_chain(vector_db.as_retriever(), combine_docs_chain)

### Generate a retrieval-augmented response to a question

Use the RAG chain to process the query. 

In [10]:
output = rag_chain.invoke({"input": "Quando Silvio Santos morreu?"})

print(output['answer'])

Silvio Santos faleceu no dia 17 de agosto de 2024, aos 93 anos.


In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template_chat = ChatPromptTemplate([
    ("system", "Você é um assistente brasileiro. Responda sempre em Português."),
    ("user", "{input}")
])


# Assemble the retrieval-augmented generation chain.
model.invoke(prompt_template_chat.format(input="Quando Silvio Santos morreu?"))

'Silvio Santos, apresentador de televisão e empresário brasileiro, está vivo e bem. Não há informações sobre a sua morte em data próxima ou distante. Portanto, essa pergunta não é relevante no momento atual.'

In [13]:
birth = "12 de dezembro de 1930"
results = { "RAG": [], "Puro": [] }
for i in range(10):
    results["RAG"].append(rag_chain.invoke({"input": "Quando Silvio Santos nasceu?"})['answer'])
    results["Puro"].append(model.invoke(prompt_template_chat.format(input="Quando Silvio Santos nasceu?")))
    
for type in ["RAG", "Puro"]:
    correct = 0
    for result in results[type]:
        print(type, "\t", result)
        correct = correct+1 if result.find(birth) != -1 else correct
    print(type, correct, "out of", len(results[type]))

RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 	 Silvio Santos nasceu em 12 de dezembro de 1930.
RAG 10 out of 10
Puro 	 Silvio Santos, conhecido também como "Homem do Brasil", foi nascido no dia 8 de outubro de 1930, na cidade de Pindorama, no estado brasileiro de São Paulo. Ele é um empresário e apresentador de televisão famoso por ter fundado o Sistema Brasileiro de Televisão (SBT), além de ser dono da Abravin, rede varejista de eletrodomésticos.
Puro 	 Silvio Santos, o famoso empresário e apresentador de televisão brasileiro, nasceu no dia 19 de j

In [14]:
rag_chain.invoke({"input": "Quando nasceu Silvio Santos?"})

{'input': 'Quando nasceu Silvio Santos?',
 'context': [Document(metadata={'pk': 455178137904611330, 'source': 'https://pt.wikipedia.org/wiki/Silvio_Santos'}, page_content='Silvio Santos, nome artístico de Senor Abravanel[7][8] (em hebraico: סניור אברבנאל;[9] Rio de Janeiro, 12 de dezembro de 1930 – São Paulo, 17 de agosto de 2024),[2] foi um apresentador de televisão, empresário, produtor, radialista e cantor brasileiro, conhecido como o "Rei da Televisão Brasileira",[10] além de ser amplamente reconhecido como a maior referência na história da comunicação do país.[11][12][13] Iniciou sua carreira de comunicador na rádio, posteriormente chegando a televisão, onde consagrou-se como uma figura central na cultura popular e entretenimento brasileiro por mais de seis décadas, conquistando o carinho do público e a expansão de seus negócios, através da fundação do Grupo Silvio Santos. Ao longo da sua vida, Silvio também esteve envolvido em outras áreas como a música e a política.[14]'),
  Doc